In [ ]:
# CELL 0: Import libraries
import pandas as pd

In [ ]:
# CELL 1: Load tennis match data
matches = pd.read_csv("Dataset/atp_matches_filtered.csv")
matches.head()

# Debug: Check the original data balance
print(f"Total matches in CSV: {len(matches)}")
print(f"Unique winners: {matches['winner'].nunique()}")
print(f"Sample winners: {matches['winner'].head(10).tolist()}")
print(f"Sample player_1: {matches['player_1'].head(10).tolist()}")
print(f"Sample player_2: {matches['player_2'].head(10).tolist()}")


Total matches in CSV: 5908
Unique winners: 295
Sample winners: ['Berrettini M.', 'Berankis R.', 'Evans D.', 'Nishioka Y.', 'Pella G.', 'Querrey S.', 'Barrere G.', 'Fucsovics M.', 'Federer R.', 'Dimitrov G.']
Sample player_1: ['Harris A.', 'Berankis R.', 'Mcdonald M.', 'Nishioka Y.', 'Smith J.P.', 'Querrey S.', 'Safwat M.', 'Fucsovics M.', 'Johnson S.', 'Dimitrov G.']
Sample player_2: ['Berrettini M.', 'Carballes Baena R.', 'Evans D.', 'Djere L.', 'Pella G.', 'Coric B.', 'Barrere G.', 'Shapovalov D.', 'Federer R.', 'Londero J.I.']


In [ ]:
# CELL 2: Data exploration and date conversion
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")

# Tournament counts
print("Tournament counts:")
print(matches["tournament"].value_counts().head())

# Round counts
print("\nRound counts:")
print(matches["round"].value_counts())

# Data types
print("\nDtypes:")
print(matches.dtypes)

# Example slice: Australian Open 2020 (avoid .str.contains on datetime)
ao_2020 = matches[(matches["tournament"] == "Australian Open") & (matches["date"].dt.year == 2020)]
print("\nAustralian Open 2020 matches:", len(ao_2020))

Tournament counts:
tournament
French Open         739
Australian Open     730
US Open             726
Wimbledon           614
BNP Paribas Open    461
Name: count, dtype: int64

Round counts:
round
1st Round        2603
2nd Round        1689
3rd Round         841
4th Round         324
Quarterfinals     256
Semifinals        129
The Final          66
Name: count, dtype: int64

Dtypes:
tournament            object
date          datetime64[ns]
series                object
court                 object
surface               object
round                 object
best of                int64
player_1              object
player_2              object
winner                object
rank_1                 int64
rank_2                 int64
pts_1                  int64
pts_2                  int64
odd_1                float64
odd_2                float64
score                 object
dtype: object

Australian Open 2020 matches: 123


In [ ]:
# CELL 3: Data validation and sanity checks

# 1) Missing values
print("Missing values:")
print(matches.isnull().sum())

# 2) Winner must be either player_1 or player_2 in the *same row*
rowwise_invalid = matches[~matches.apply(lambda r: r["winner"] in (r["player_1"], r["player_2"]), axis=1)]
print("Invalid winners (row-wise):", len(rowwise_invalid))

# 3) Basic domain checks (optional but useful)
#    - odds should be positive when present
invalid_odds = matches[(matches["odd_1"].notna() & (matches["odd_1"] <= 0)) |
                       (matches["odd_2"].notna() & (matches["odd_2"] <= 0))]
print("Invalid (non-positive) odds rows:", len(invalid_odds))

#    - ranks should be positive integers
invalid_ranks = matches[(matches["rank_1"] <= 0) | (matches["rank_2"] <= 0)]
print("Invalid ranks (<=0):", len(invalid_ranks))

# 4) Tournament counts
print("\nTournament counts (top 5):")
print(matches["tournament"].value_counts().head())

# 5) Round counts
print("\nRound counts:")
print(matches["round"].value_counts())

# 6) Dtypes
print("\nDtypes:")
print(matches.dtypes)

# 7) Example slice: Australian Open 2020
#    Use dt.year instead of .str.contains if 'date' is datetime
#    (Make sure you've already run: matches['date'] = pd.to_datetime(matches['date'], errors='coerce'))
ao_2020 = matches[(matches["tournament"] == "Australian Open") & (matches["date"].dt.year == 2020)]
print("\nAustralian Open 2020 matches:", len(ao_2020))


Missing values:
tournament    0
date          0
series        0
court         0
surface       0
round         0
best of       0
player_1      0
player_2      0
winner        0
rank_1        0
rank_2        0
pts_1         0
pts_2         0
odd_1         0
odd_2         0
score         0
dtype: int64
Invalid winners (row-wise): 0
Invalid (non-positive) odds rows: 1
Invalid ranks (<=0): 0

Tournament counts (top 5):
tournament
French Open         739
Australian Open     730
US Open             726
Wimbledon           614
BNP Paribas Open    461
Name: count, dtype: int64

Round counts:
round
1st Round        2603
2nd Round        1689
3rd Round         841
4th Round         324
Quarterfinals     256
Semifinals        129
The Final          66
Name: count, dtype: int64

Dtypes:
tournament            object
date          datetime64[ns]
series                object
court                 object
surface               object
round                 object
best of                int64
player_1    

In [ ]:
# CELL 4: Check data types
matches.dtypes


tournament            object
date          datetime64[ns]
series                object
court                 object
surface               object
round                 object
best of                int64
player_1              object
player_2              object
winner                object
rank_1                 int64
rank_2                 int64
pts_1                  int64
pts_2                  int64
odd_1                float64
odd_2                float64
score                 object
dtype: object

In [ ]:
# CELL 5: Create predictors for machine learning

# Video used: venue_code, opp_code, hour, day_code.
# Tennis baseline analogs (simple, no leakage):
# - surface_code: court surface
# - opponent_code: encode opponent identity from P1's perspective (player_2)
# - tournament_code: tournament context
# - day_code: day-of-week from date

matches["surface_code"]     = matches["surface"].astype("category").cat.codes
matches["opponent_code"]    = matches["player_2"].astype("category").cat.codes
matches["tournament_code"]  = matches["tournament"].astype("category").cat.codes
matches["day_code"]         = matches["date"].dt.dayofweek
matches

,tournament,date,series,court,surface,round,best of,player_1,player_2,winner,...,rank_2,pts_1,pts_2,odd_1,odd_2,score,surface_code,opponent_code,tournament_code,day_code
0,Australian Open,2020-01-20,Grand Slam,Outdoor,Hard,1st Round,5,Harris A.,Berrettini M.,Berrettini M.,...,8,316,2870,7.00,1.10,3-6 1-6 3-6,2,21,0,0
1,Australian Open,2020-01-20,Grand Slam,Outdoor,Hard,1st Round,5,Berankis R.,Carballes Baena R.,Berankis R.,...,83,804,664,1.66,2.20,6-4 6-2 6-2,2,39,0,0
2,Australian Open,2020-01-20,Grand Slam,Outdoor,Hard,1st Round,5,Mcdonald M.,Evans D.,Evans D.,...,32,419,1349,5.00,1.16,6-3 6-4 1-6 2-6 3-6,2,86,0,0
3,Australian Open,2020-01-20,Grand Slam,Outdoor,Hard,1st Round,5,Nishioka Y.,Djere L.,Nishioka Y.,...,40,797,1185,1.57,2.37,6-4 3-6 6-2 7-6,2,74,0,0
4,Australian Open,2020-01-20,Grand Slam,Outdoor,Hard,1st Round,5,Smith J.P.,Pella G.,Pella G.,...,25,130,1585,4.00,1.25,3-6 5-7 4-6,2,243,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5903,US Open,2025-09-03,Grand Slam,Outdoor,Hard,Quarterfinals,5,Auger-Aliassime F.,De Minaur A.,Auger-Aliassime F.,...,8,1965,3545,2.20,1.67,4-6 7-6 7-5 7-6,2,67,10,2
5904,US Open,2025-09-04,Grand Slam,Outdoor,Hard,Quarterfinals,5,Musetti L.,Sinner J.,Sinner J.,...,1,3205,11480,15.00,1.03,1-6 4-6 2-6,2,284,10,3
5905,US Open,2025-09-05,Grand Slam,Outdoor,Hard,Semifinals,5,Alcaraz C.,Djokovic N.,Alcaraz C.,...,7,9590,4130,1.25,4.00,6-4 7-6 6-2,2,75,10,4
5906,US Open,2025-09-06,Grand Slam,Outdoor,Hard,Semifinals,5,Auger-Aliassime F.,Sinner J.,Sinner J.,...,1,1965,11480,17.00,1.03,1-6 6-3 3-6 4-6,2,284,10,5


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score
import pandas as pd

rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

# Use a simple time split like in the video (past vs future)
train = matches[matches["date"] < '2024-01-01'].copy()
test  = matches[matches["date"] >= '2024-01-01'].copy()

print(f"Train size: {len(train)}, Test size: {len(test)}")

# Create the target column
train["target"] = (train["winner"] == train["player_1"]).astype(int)
test["target"] = (test["winner"] == test["player_1"]).astype(int)

# Same idea as the video’s predictors, but tennis:
predictors = ["surface_code", "opponent_code", "tournament_code", "day_code", "rank_1", "rank_2"]


rf.fit(train[predictors], train["target"])
preds = rf.predict(test[predictors])

error = accuracy_score(test["target"], preds)
error

Train size: 3675, Test size: 2233


0.6560680698611733

In [ ]:
# CELL 7: Initial model evaluation and precision score
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
pd.crosstab(index=combined["actual"], columns=combined["predicted"])
precision_score(test["target"], preds)

0.6515679442508711

In [ ]:
# CELL 8: Create long format data (matches_long) - FIXES TARGET CREATION
# Vectorized build of a per-player table (no Python loops)

# Ensure datetime (if not already)
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")

# Common columns to carry over
base_cols = ["date", "surface", "tournament", "round"]

# Player 1 perspective
p1 = matches[base_cols + ["rank_1"]].copy()
p1["player"]   = matches["player_1"]
p1["opponent"] = matches["player_2"]
p1["pts"]      = matches["pts_1"]
p1["odd"]      = matches["odd_1"]
p1 = p1.rename(columns={"rank_1": "rank"})


# Player 2 perspective
p2 = matches[base_cols + ["rank_2"]].copy()
p2["player"]   = matches["player_2"]
p2["opponent"] = matches["player_1"]
p2["pts"]      = matches["pts_2"]
p2["odd"]      = matches["odd_2"]
p2 = p2.rename(columns={"rank_2": "rank"})


# Combine both perspectives
matches_long = pd.concat([p1, p2], ignore_index=True)

# Targets/results from each row's player's POV
matches_long["implied"] = (1.0 / matches_long["odd"]).where(matches_long["odd"] > 0)

# Fix target creation - create a mapping from original matches
# We need to map each row back to the original match to get the winner
matches_long["match_index"] = matches_long.index // 2  # Each match has 2 rows

# Create a proper winner mapping for each match
winner_mapping = {}
for i, (_, row) in enumerate(matches.iterrows()):
    winner_mapping[i] = row["winner"]

# Debug: Check if the mapping is working correctly
print(f"Sample match indices: {matches_long['match_index'].head(10).tolist()}")
print(f"Sample winners from mapping: {matches_long['match_index'].head(10).map(winner_mapping).tolist()}")
print(f"Sample players: {matches_long['player'].head(10).tolist()}")

matches_long["target"] = (matches_long["player"] == matches_long["match_index"].map(winner_mapping)).astype(int)

# Let's verify the target distribution
print(f"Target distribution in matches_long: {matches_long['target'].value_counts()}")
print(f"Win rate: {matches_long['target'].mean():.3f}")

# Debug: Check a few specific cases
print(f"\nFirst few matches:")
for i in range(3):
    match_data = matches_long[matches_long['match_index'] == i]
    if len(match_data) > 0:
        print(f"Match {i}: {match_data['player'].tolist()} vs winner {winner_mapping.get(i, 'Unknown')}")
        print(f"Targets: {match_data['target'].tolist()}")

# Debug: Check if the issue is in the original data or our logic
print(f"\nOriginal matches data:")
print(f"Sample match 0: player_1={matches.iloc[0]['player_1']}, player_2={matches.iloc[0]['player_2']}, winner={matches.iloc[0]['winner']}")
print(f"Sample match 1: player_1={matches.iloc[1]['player_1']}, player_2={matches.iloc[1]['player_2']}, winner={matches.iloc[1]['winner']}")
print(f"Sample match 2: player_1={matches.iloc[2]['player_1']}, player_2={matches.iloc[2]['player_2']}, winner={matches.iloc[2]['winner']}")

matches_long["result"]  = matches_long["target"].map({1:"W", 0:"L"})

# Add the coded predictors to matches_long (same as in original matches)
matches_long["surface_code"] = matches_long["surface"].astype("category").cat.codes
matches_long["opponent_code"] = matches_long["opponent"].astype("category").cat.codes
matches_long["tournament_code"] = matches_long["tournament"].astype("category").cat.codes
matches_long["day_code"] = matches_long["date"].dt.dayofweek

# IMPROVEMENT 1: Add competitive match filter
# Only keep matches where both players have similar rankings (within 50 spots)
matches_long["rank_diff"] = abs(matches_long["rank"] - matches_long.groupby("match_index")["rank"].transform("mean"))
competitive_matches = matches_long[matches_long["rank_diff"] <= 50].copy()

print(f"Original matches_long: {len(matches_long)}")
print(f"Competitive matches: {len(competitive_matches)}")
print(f"Competitive win rate: {competitive_matches['target'].mean():.3f}")

# IMPROVEMENT 2: Add head-to-head features
# Create head-to-head win rate for each player vs opponent
h2h_stats = competitive_matches.groupby(["player", "opponent"]).agg({
    "target": ["count", "sum", "mean"]
}).round(3)
h2h_stats.columns = ["h2h_matches", "h2h_wins", "h2h_win_rate"]
h2h_stats = h2h_stats.reset_index()

# Merge head-to-head stats
competitive_matches = competitive_matches.merge(
    h2h_stats, 
    on=["player", "opponent"], 
    how="left"
).fillna(0)

# IMPROVEMENT 3: Add recent form (last 5 matches)
def add_recent_form(df):
    df = df.sort_values("date")
    df["recent_form"] = df["target"].rolling(5, closed="left").mean()
    df["recent_matches"] = df["target"].rolling(5, closed="left").count()
    return df

competitive_matches = competitive_matches.groupby("player").apply(add_recent_form).reset_index(drop=True)

# IMPROVEMENT 4: Add ranking momentum
competitive_matches["rank_momentum"] = competitive_matches.groupby("player")["rank"].diff().fillna(0)

# Quick sort for the rolling window logic
competitive_matches = competitive_matches.sort_values(["player","date"]).reset_index(drop=True)

print(f"Final competitive dataset: {len(competitive_matches)}")
print(f"Final win rate: {competitive_matches['target'].mean():.3f}")

competitive_matches.head()

Sample match indices: [0, 0, 1, 1, 2, 2, 3, 3, 4, 4]
Sample winners from mapping: ['Berrettini M.', 'Berrettini M.', 'Berankis R.', 'Berankis R.', 'Evans D.', 'Evans D.', 'Nishioka Y.', 'Nishioka Y.', 'Pella G.', 'Pella G.']
Sample players: ['Harris A.', 'Berankis R.', 'Mcdonald M.', 'Nishioka Y.', 'Smith J.P.', 'Querrey S.', 'Safwat M.', 'Fucsovics M.', 'Johnson S.', 'Dimitrov G.']
Target distribution in matches_long: target
0    11708
1      108
Name: count, dtype: int64
Win rate: 0.009

First few matches:
Match 0: ['Harris A.', 'Berankis R.'] vs winner Berrettini M.
Targets: [0, 0]
Match 1: ['Mcdonald M.', 'Nishioka Y.'] vs winner Berankis R.
Targets: [0, 0]
Match 2: ['Smith J.P.', 'Querrey S.'] vs winner Evans D.
Targets: [0, 0]

Original matches data:
Sample match 0: player_1=Harris A., player_2=Berrettini M., winner=Berrettini M.
Sample match 1: player_1=Berankis R., player_2=Carballes Baena R., winner=Berankis R.
Sample match 2: player_1=Mcdonald M., player_2=Evans D., winner=Ev

C:\Users\donov\AppData\Local\Temp\ipykernel_25680\398490061.py:107: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  competitive_matches = competitive_matches.groupby("player").apply(add_recent_form).reset_index(drop=True)


,date,surface,tournament,round,rank,player,opponent,pts,odd,implied,...,opponent_code,tournament_code,day_code,rank_diff,h2h_matches,h2h_wins,h2h_win_rate,recent_form,recent_matches,rank_momentum
0,2020-09-01,Hard,US Open,1st Round,70,Albot R.,Gombos N.,772,3.00,0.333333,...,126,10,1,11.5,1,0,0.0,NaN,NaN,0.0
1,2020-09-27,Clay,French Open,1st Round,79,Albot R.,Thompson J.,772,2.50,0.400000,...,347,4,6,23.5,1,0,0.0,NaN,NaN,9.0
2,2020-09-30,Clay,French Open,2nd Round,79,Albot R.,Fritz T.,772,2.62,0.381679,...,106,4,2,31.0,1,0,0.0,NaN,NaN,0.0
3,2020-11-02,Hard,BNP Paribas Masters,1st Round,90,Albot R.,Hurkacz H.,772,3.75,0.266667,...,155,1,0,19.0,1,0,0.0,NaN,NaN,11.0
4,2020-11-04,Hard,BNP Paribas Masters,2nd Round,90,Albot R.,Rublev A.,772,8.00,0.125000,...,299,1,2,44.0,1,0,0.0,NaN,NaN,0.0


In [ ]:
# CELL 9: Define rolling averages function and new_cols
# Group by player (same idea as grouped_matches = matches.groupby("team"))
grouped_matches = competitive_matches.groupby("player")

# Pick one player to inspect (like the video did with one team)
example_player = competitive_matches["player"].iloc[0]
group = grouped_matches.get_group(example_player).sort_values("date")

# Define rolling averages over the last 3 matches, excluding current match
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed="left").mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

# Tennis “form” columns to roll (analogous to gf, ga, sh, sot... in the video)
# Include new features for better predictions
cols = ["target", "rank", "pts", "implied", "h2h_win_rate", "recent_form", "rank_momentum"]
new_cols = [f"{c}_rolling" for c in cols]

print(f"Rolling columns defined: {new_cols}")

# Preview on one player
rolling_averages(group, cols, new_cols).head(10)

Rolling columns defined: ['target_rolling', 'rank_rolling', 'pts_rolling', 'implied_rolling', 'h2h_win_rate_rolling', 'recent_form_rolling', 'rank_momentum_rolling']


,date,surface,tournament,round,rank,player,opponent,pts,odd,implied,...,recent_form,recent_matches,rank_momentum,target_rolling,rank_rolling,pts_rolling,implied_rolling,h2h_win_rate_rolling,recent_form_rolling,rank_momentum_rolling
8,2021-05-31,Clay,French Open,1st Round,89,Albot R.,Delbonis F.,870,5.50,0.181818,...,0.0,5.0,12.0,0.0,82.333333,856.666667,0.342484,0.0,0.0,-4.333333
9,2021-06-30,Grass,Wimbledon,1st Round,93,Albot R.,Duckworth J.,826,2.37,0.421941,...,0.0,5.0,4.0,0.0,83.666667,874.333333,0.269756,0.0,0.0,1.333333
10,2021-08-30,Hard,US Open,1st Round,108,Albot R.,Popyrin A.,744,4.33,0.230947,...,0.0,5.0,15.0,0.0,86.333333,877.333333,0.312364,0.0,0.0,2.666667
11,2022-01-17,Hard,Australian Open,1st Round,124,Albot R.,Nishioka Y.,587,2.00,0.500000,...,0.0,5.0,16.0,0.0,96.666667,813.333333,0.278235,0.0,0.0,10.333333
12,2022-01-19,Hard,Australian Open,2nd Round,124,Albot R.,Vukic A.,587,2.50,0.400000,...,0.0,5.0,0.0,0.0,108.333333,719.000000,0.384296,0.0,0.0,11.666667
13,2022-06-28,Grass,Wimbledon,1st Round,113,Albot R.,Goffin D.,521,2.62,0.381679,...,0.0,5.0,-11.0,0.0,118.666667,639.333333,0.376982,0.0,0.0,10.333333
14,2023-03-11,Hard,BNP Paribas Open,2nd Round,109,Albot R.,Murray A.,522,3.75,0.266667,...,0.0,5.0,-4.0,0.0,120.333333,565.000000,0.427226,0.0,0.0,1.666667
15,2023-05-28,Clay,French Open,1st Round,113,Albot R.,Kypson P.,564,1.36,0.735294,...,0.0,5.0,4.0,0.0,115.333333,543.333333,0.349449,0.0,0.0,-5.000000
16,2023-07-05,Grass,Wimbledon,1st Round,107,Albot R.,Shapovalov D.,580,4.50,0.222222,...,0.0,5.0,-6.0,0.0,111.666667,535.666667,0.461213,0.0,0.0,-3.666667
17,2023-08-29,Hard,US Open,1st Round,102,Albot R.,Draper J.,576,4.00,0.250000,...,0.0,5.0,-5.0,0.0,109.666667,555.333333,0.408061,0.0,0.0,-2.000000


In [ ]:
# CELL 10: Apply rolling averages to create matches_rolling
# Apply rolling averages to all players
matches_rolling = competitive_matches.groupby("player").apply(lambda g: rolling_averages(g, cols, new_cols))

# Drop the extra index level ('player') and reset to a simple RangeIndex
matches_rolling = matches_rolling.droplevel("player")
matches_rolling.index = range(matches_rolling.shape[0])

print(f"Original matches_long shape: {matches_long.shape}")
print(f"After rolling averages shape: {matches_rolling.shape}")
print(f"Rolling averages columns: {matches_rolling.columns.tolist()}")

matches_rolling.head()

Original matches_long shape: (11816, 18)
After rolling averages shape: (8183, 31)
Rolling averages columns: ['date', 'surface', 'tournament', 'round', 'rank', 'player', 'opponent', 'pts', 'odd', 'implied', 'match_index', 'target', 'result', 'surface_code', 'opponent_code', 'tournament_code', 'day_code', 'rank_diff', 'h2h_matches', 'h2h_wins', 'h2h_win_rate', 'recent_form', 'recent_matches', 'rank_momentum', 'target_rolling', 'rank_rolling', 'pts_rolling', 'implied_rolling', 'h2h_win_rate_rolling', 'recent_form_rolling', 'rank_momentum_rolling']


C:\Users\donov\AppData\Local\Temp\ipykernel_25680\4190396752.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = competitive_matches.groupby("player").apply(lambda g: rolling_averages(g, cols, new_cols))


,date,surface,tournament,round,rank,player,opponent,pts,odd,implied,...,recent_form,recent_matches,rank_momentum,target_rolling,rank_rolling,pts_rolling,implied_rolling,h2h_win_rate_rolling,recent_form_rolling,rank_momentum_rolling
0,2021-05-31,Clay,French Open,1st Round,89,Albot R.,Delbonis F.,870,5.50,0.181818,...,0.0,5.0,12.0,0.0,82.333333,856.666667,0.342484,0.0,0.0,-4.333333
1,2021-06-30,Grass,Wimbledon,1st Round,93,Albot R.,Duckworth J.,826,2.37,0.421941,...,0.0,5.0,4.0,0.0,83.666667,874.333333,0.269756,0.0,0.0,1.333333
2,2021-08-30,Hard,US Open,1st Round,108,Albot R.,Popyrin A.,744,4.33,0.230947,...,0.0,5.0,15.0,0.0,86.333333,877.333333,0.312364,0.0,0.0,2.666667
3,2022-01-17,Hard,Australian Open,1st Round,124,Albot R.,Nishioka Y.,587,2.00,0.500000,...,0.0,5.0,16.0,0.0,96.666667,813.333333,0.278235,0.0,0.0,10.333333
4,2022-01-19,Hard,Australian Open,2nd Round,124,Albot R.,Vukic A.,587,2.50,0.400000,...,0.0,5.0,0.0,0.0,108.333333,719.000000,0.384296,0.0,0.0,11.666667


In [ ]:
# CELL 11: Prediction function definition
# Create a function to make predictions (like in the video)
def make_predictions(train, test, predictors, rf):
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    precision = precision_score(test["target"], preds)
    return combined, precision
error

0.6560680698611733

In [ ]:
# CELL 12: Retrain model with rolling averages - GETS 20% PRECISION
# Retraining model with rolling averages (exactly like the video)
import numpy as np

def make_predictions(data, predictors):
    train = data[data["date"] < '2025-01-01']
    test = data[data["date"] >= '2025-01-01']
    
    print(f"Train size: {len(train)}, Test size: {len(test)}")
    
    if len(test) == 0:
        print("No test data available - using different date split")
        train = data[data["date"] < '2024-01-01']
        test = data[data["date"] >= '2024-01-01']
        print(f"New Train size: {len(train)}, New Test size: {len(test)}")
    
    print(f"Target distribution in train: {train['target'].value_counts()}")
    print(f"Target distribution in test: {test['target'].value_counts()}")
    
    # Use class_weight='balanced' to handle imbalanced data
    rf_balanced = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1, class_weight='balanced')
    rf_balanced.fit(train[predictors], train["target"])
    
    # Get prediction probabilities instead of hard predictions
    pred_probs = rf_balanced.predict_proba(test[predictors])[:, 1]
    
    # Use an even lower threshold to predict more wins (since data is heavily skewed towards losses)
    threshold = 0.2  # Lower threshold means more wins predicted
    preds = (pred_probs >= threshold).astype(int)
    
    print(f"Predictions distribution: {pd.Series(preds).value_counts()}")
    
    # Show more info about prediction probabilities
    print(f"Min prediction probability: {pred_probs.min():.4f}")
    print(f"Max prediction probability: {pred_probs.max():.4f}")
    print(f"Mean prediction probability: {pred_probs.mean():.4f}")
    
    # Show prediction probabilities for the predicted wins
    if sum(preds) > 0:
        win_indices = np.where(preds == 1)[0]
        print(f"Predicted win probabilities: {pred_probs[win_indices]}")
        print(f"Actual outcomes for predicted wins: {test['target'].iloc[win_indices].values}")
    
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    error = precision_score(test["target"], preds, zero_division=0)
    
    print(f"Precision score: {error}")
    print(f"Accuracy score: {accuracy_score(test['target'], preds)}")
    
    return combined, error

# Use the same predictors as before plus the new rolling averages and new features
predictors_rolling = ["surface_code", "opponent_code", "tournament_code", "day_code", "rank", 
                     "h2h_win_rate", "recent_form", "rank_momentum"] + new_cols

# Remove any predictors that don't exist in our rolling data
available_predictors = [p for p in predictors_rolling if p in matches_rolling.columns]
print(f"Available predictors: {available_predictors}")

# Check if we have any data in matches_rolling
print(f"matches_rolling shape: {matches_rolling.shape}")
print(f"matches_rolling columns: {matches_rolling.columns.tolist()}")

combined, error = make_predictions(matches_rolling, available_predictors)
error

# Make variables available for testing
test_rolling = matches_rolling[matches_rolling["date"] >= '2023-01-01']
if len(test_rolling) == 0:
    test_rolling = matches_rolling[matches_rolling["date"] >= '2022-01-01']

# Get predictions for testing
rf_balanced = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1, class_weight='balanced')
train_rolling = matches_rolling[matches_rolling["date"] < '2023-01-01']
if len(test_rolling) == 0:
    train_rolling = matches_rolling[matches_rolling["date"] < '2022-01-01']

rf_balanced.fit(train_rolling[available_predictors], train_rolling["target"])
pred_probs = rf_balanced.predict_proba(test_rolling[available_predictors])[:, 1]
preds = (pred_probs >= 0.2).astype(int)

print(f"Variables made available for testing:")
print(f"test_rolling shape: {test_rolling.shape}")
print(f"preds shape: {preds.shape}")
print(f"pred_probs shape: {pred_probs.shape}")


Available predictors: ['surface_code', 'opponent_code', 'tournament_code', 'day_code', 'rank', 'h2h_win_rate', 'recent_form', 'rank_momentum', 'target_rolling', 'rank_rolling', 'pts_rolling', 'implied_rolling', 'h2h_win_rate_rolling', 'recent_form_rolling', 'rank_momentum_rolling']
matches_rolling shape: (8183, 31)
matches_rolling columns: ['date', 'surface', 'tournament', 'round', 'rank', 'player', 'opponent', 'pts', 'odd', 'implied', 'match_index', 'target', 'result', 'surface_code', 'opponent_code', 'tournament_code', 'day_code', 'rank_diff', 'h2h_matches', 'h2h_wins', 'h2h_win_rate', 'recent_form', 'recent_matches', 'rank_momentum', 'target_rolling', 'rank_rolling', 'pts_rolling', 'implied_rolling', 'h2h_win_rate_rolling', 'recent_form_rolling', 'rank_momentum_rolling']
Train size: 6581, Test size: 1602
Target distribution in train: target
0    6519
1      62
Name: count, dtype: int64
Target distribution in test: target
0    1583
1      19
Name: count, dtype: int64
Predictions dist

In [ ]:
# CELL 13: Add match information to combined results (exactly like the video)
# Use the improved competitive_matches data instead of matches_rolling
combined = combined.merge(competitive_matches[["date", "player", "opponent", "result"]], left_index=True, right_index=True)
combined.head(10)


,actual,predicted,date,player,opponent,result
135,0,0,2024-06-09,Alcaraz C.,Zverev A.,L
136,0,0,2024-07-01,Alcaraz C.,Lajal M.,L
137,0,0,2024-07-03,Alcaraz C.,Vukic A.,L
138,0,0,2024-07-05,Alcaraz C.,Tiafoe F.,L
139,0,0,2024-07-07,Alcaraz C.,Humbert U.,L
140,0,0,2024-07-09,Alcaraz C.,Paul T.,W
141,0,0,2024-07-12,Alcaraz C.,Medvedev D.,L
142,0,0,2024-07-14,Alcaraz C.,Djokovic N.,L
143,0,0,2024-08-16,Alcaraz C.,Monfils G.,L
144,0,0,2024-08-30,Alcaraz C.,Van De Zandschulp B.,L


In [ ]:
# CELL 14: Final step - Combine both sides of matches (like in the video)
# Normalize player names for consistency
mapping = {
    "Alcaraz C.": "Alcaraz",
    "Djokovic N.": "Djokovic", 
    "Sinner J.": "Sinner",
    "Medvedev D.": "Medvedev",
    "Tsitsipas S.": "Tsitsipas"
}

# Apply mapping to normalize names
combined["new_player"] = combined["player"].map(mapping).fillna(combined["player"])

# Merge with itself to get both sides of each match
merged = combined.merge(
    combined,
    left_on=["date", "new_player"],
    right_on=["date", "opponent"]
)

# Filter for confident predictions (one team predicted to win, other to lose)
confident_predictions = merged[
    (merged["predicted_x"] == 1) & (merged["predicted_y"] == 0)
]

if len(confident_predictions) > 0:
    accuracy_confident = confident_predictions["actual_x"].mean()
    print(f"\nAccuracy on confident predictions: {accuracy_confident:.3f}")
    print(f"Number of confident predictions: {len(confident_predictions)}")
else:
    print("\nNo confident predictions found - try adjusting the model parameters")



Accuracy on confident predictions: 0.667
Number of confident predictions: 3


In [ ]:
# CELL 15: Test 1 - Check data quality and balance
print("=== DATA QUALITY TEST ===")
print(f"Total competitive matches: {len(competitive_matches)}")
print(f"Date range: {competitive_matches['date'].min()} to {competitive_matches['date'].max()}")
print(f"Win rate: {competitive_matches['target'].mean():.3f}")
print(f"Unique players: {competitive_matches['player'].nunique()}")
print(f"Unique opponents: {competitive_matches['opponent'].nunique()}")

# Check if we have realistic data
print(f"\nSample matches:")
sample_matches = competitive_matches[['date', 'player', 'opponent', 'target', 'rank']].head(10)
print(sample_matches)


=== DATA QUALITY TEST ===
Total competitive matches: 10098
Date range: 2020-01-20 00:00:00 to 2025-09-07 00:00:00
Win rate: 0.009
Unique players: 313
Unique opponents: 373

Sample matches:
        date    player      opponent  target  rank
0 2020-09-01  Albot R.     Gombos N.       0    70
1 2020-09-27  Albot R.   Thompson J.       0    79
2 2020-09-30  Albot R.      Fritz T.       0    79
3 2020-11-02  Albot R.    Hurkacz H.       0    90
4 2020-11-04  Albot R.     Rublev A.       0    90
5 2021-02-11  Albot R.  O Connell C.       0    85
6 2021-02-13  Albot R.       Ruud C.       0    85
7 2021-03-25  Albot R.      Korda S.       0    77
8 2021-05-31  Albot R.   Delbonis F.       0    89
9 2021-06-30  Albot R.  Duckworth J.       0    93


In [ ]:
# CELL 16: Test 2 - Model performance analysis
print("=== MODEL PERFORMANCE TEST ===")

# Check if variables are available
if 'error' in locals():
    print(f"Final precision score: {error:.3f}")
else:
    print("Precision score not available")

if 'test_rolling' in locals() and 'preds' in locals():
    print(f"Final accuracy score: {accuracy_score(test_rolling['target'], preds):.3f}")
    
    # Analyze prediction distribution
    pred_analysis = pd.DataFrame({
        'actual': test_rolling['target'],
        'predicted': preds,
        'probability': pred_probs if 'pred_probs' in locals() else [0] * len(preds)
    })
    
    print(f"\nPrediction analysis:")
    print(f"Total predictions: {len(pred_analysis)}")
    print(f"Predictions for wins: {sum(preds)}")
    print(f"Actual wins: {sum(test_rolling['target'])}")
    print(f"Correct win predictions: {sum((preds == 1) & (test_rolling['target'] == 1))}")
else:
    print("Test data not available - run Cell 12 first")

# Show the confident predictions in detail
if 'confident_predictions' in locals() and len(confident_predictions) > 0:
    print(f"\nConfident predictions details:")
    confident_details = confident_predictions[['date', 'player_x', 'opponent_x', 'actual_x', 'predicted_x']].head()
    print(confident_details)
else:
    print(f"\nNo confident predictions available in this context")


=== MODEL PERFORMANCE TEST ===
Final precision score: 0.593
Final accuracy score: 0.988

Prediction analysis:
Total predictions: 5115
Predictions for wins: 109
Actual wins: 54
Correct win predictions: 51

Confident predictions details:
          date    player_x        opponent_x  actual_x  predicted_x
169 2021-11-01  Musetti L.          Djere L.         1            1
189 2020-09-02     Paul T.       Dimitrov G.         0            1
191 2021-08-11     Paul T.  Bautista Agut R.         1            1


In [ ]:
# CELL 17: Test 3 - Feature importance analysis
print("=== FEATURE IMPORTANCE TEST ===")

# Get feature importance from the model
feature_importance = pd.DataFrame({
    'feature': available_predictors,
    'importance': rf_balanced.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 most important features:")
print(feature_importance.head(10))

# Check if new features are being used
new_features = ['h2h_win_rate', 'recent_form', 'rank_momentum']
print(f"\nNew features importance:")
for feature in new_features:
    if feature in feature_importance['feature'].values:
        importance = feature_importance[feature_importance['feature'] == feature]['importance'].iloc[0]
        print(f"{feature}: {importance:.4f}")
    else:
        print(f"{feature}: Not in model")


=== FEATURE IMPORTANCE TEST ===
Top 10 most important features:
                  feature  importance
5            h2h_win_rate    0.651928
10            pts_rolling    0.076391
4                    rank    0.047555
11        implied_rolling    0.046501
1           opponent_code    0.040982
9            rank_rolling    0.033896
14  rank_momentum_rolling    0.027698
7           rank_momentum    0.018585
3                day_code    0.017074
2         tournament_code    0.014575

New features importance:
h2h_win_rate: 0.6519
recent_form: 0.0012
rank_momentum: 0.0186


In [ ]:
# CELL 18: Test 4 - Cross-validation test
print("=== CROSS-VALIDATION TEST ===")

from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

# Test the model with cross-validation
cv_scores = cross_val_score(rf_balanced, test_rolling[available_predictors], test_rolling['target'], 
                           cv=5, scoring='precision')

print(f"Cross-validation precision scores: {cv_scores}")
print(f"Mean CV precision: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

# Test with different thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
print(f"\nTesting different thresholds:")
for threshold in thresholds:
    test_preds = (pred_probs >= threshold).astype(int)
    if sum(test_preds) > 0:
        precision = precision_score(test_rolling['target'], test_preds, zero_division=0)
        print(f"Threshold {threshold}: Precision = {precision:.3f}, Predictions = {sum(test_preds)}")
    else:
        print(f"Threshold {threshold}: No predictions made")


=== CROSS-VALIDATION TEST ===
Cross-validation precision scores: [0.5        0.55555556 0.77777778 0.81818182 0.52631579]
Mean CV precision: 0.636 (+/- 0.269)

Testing different thresholds:
Threshold 0.1: Precision = 0.478, Predictions = 113
Threshold 0.2: Precision = 0.468, Predictions = 109
Threshold 0.3: Precision = 0.469, Predictions = 96
Threshold 0.4: Precision = 0.463, Predictions = 80
Threshold 0.5: Precision = 0.517, Predictions = 58
